<a href="https://colab.research.google.com/github/keyvanmahmoudi/LLM_Engineering_Course/blob/main/solutions/Session09_evaluating_and_improving_rag_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۹: ارزیابی و بهبود RAG

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session09_evaluating_and_improving_rag.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
!pip -q install openai sentence-transformers rank_bm25 hazm
!wget -q -N https://raw.githubusercontent.com/keyvanmahmoudi/LLM_Engineering_Course/main/llm_utils.py

from llm_utils import setup_llm, chat

client, MODEL = setup_llm("ollama")      # or: setup_llm("api")

In [ ]:
import json

import numpy as np
import pandas as pd
from hazm import Normalizer, word_tokenize
from llm_utils import download_data
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

chunks = [json.loads(line) for line in open(download_data("novinkala_chunks.jsonl"), encoding="utf-8")]
eval_set = [json.loads(line) for line in open(download_data("rag_eval.jsonl"), encoding="utf-8")]
_normalizer = Normalizer(persian_numbers=False)
_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def normalize(text):
    return _normalizer.normalize(text).translate(_DIGITS)


def evaluate_retrieval(search_fn, questions=None, ks=(1, 3, 5)):
    questions = questions or eval_set
    hits, rr = {k: [] for k in ks}, []
    for item in questions:
        evidence = normalize(item["evidence"])
        ranking = search_fn(item["question"])
        relevant = [i for i, idx in enumerate(ranking) if evidence in chunks[idx]["text"]]
        first = relevant[0] + 1 if relevant else None
        rr.append(1 / first if first else 0)
        for k in ks:
            hits[k].append(first is not None and first <= k)
    return {**{f"hit@{k}": np.mean(v) for k, v in hits.items()}, "MRR": np.mean(rr)}


embedder = SentenceTransformer("intfloat/multilingual-e5-base")
chunk_vectors = embedder.encode([f"passage: {c['text']}" for c in chunks], normalize_embeddings=True)


def dense_scores(query):
    return chunk_vectors @ embedder.encode([f"query: {normalize(query)}"], normalize_embeddings=True)[0]


STOPWORDS = set("و در به از که را با این آن است برای یا تا هم یک بر اگر هر شود می‌شود شده".split())


def make_bm25(tokenizer, stopwords):
    def tokenize(text):
        return [t for t in tokenizer(normalize(text)) if t not in stopwords and len(t) > 1]
    index = BM25Okapi([tokenize(c["text"]) for c in chunks])
    return lambda query: index.get_scores(tokenize(query))


bm25_scores = make_bm25(word_tokenize, STOPWORDS)


def rank(scores, n=20):
    return list(np.argsort(-scores)[:n])


def rrf(rankings, c=60):
    fused = {}
    for ranking in rankings:
        for r, idx in enumerate(ranking, 1):
            fused[idx] = fused.get(idx, 0) + 1 / (c + r)
    return sorted(fused, key=fused.get, reverse=True)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

فهرست Stopwordها را حذف کنید و BM25 را دوباره ارزیابی کنید. سپس به جای `word_tokenize` از `split()` ساده استفاده کنید. کدام تغییر اثر بیشتری دارد؟

</div>

In [ ]:
variants = {
    "hazm + stopwords": make_bm25(word_tokenize, STOPWORDS),
    "hazm, no stopwords": make_bm25(word_tokenize, set()),
    "split + stopwords": make_bm25(str.split, STOPWORDS),
    "split, no stopwords": make_bm25(str.split, set()),
}
pd.DataFrame({name: evaluate_retrieval(lambda q, f=fn: rank(f(q))) for name, fn in variants.items()}).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

تفاوت اصلی `word_tokenize` و `split` در **علائم نگارشی** است. با `split`، کلمه‌ای که به علامت چسبیده، مثل «است.» یا «دارد؟»، توکنی جدا از «است» و «دارد» حساب می‌شود و با پرسش جور نمی‌شود. پس معمولاً تغییر توکن‌ساز اثر بیشتری از حذف Stopwordها دارد.

BM25 به‌خاطر وزن IDF، خودش کلمه‌های بسیار رایج را کم‌اهمیت می‌کند. به همین دلیل اثر حذف Stopwordها در BM25 کمتر از روش ساده‌ای است که فقط کلمه‌های مشترک را می‌شمارد. یک نکتهٔ دیگر: hazm نیم‌فاصله را حفظ می‌کند، پس «می‌شود» یک توکن است. ولی اگر کاربر «میشه» یا «می شود» بنویسد، با سند جور نمی‌شود. این کمبود را جستجوی معنایی جبران می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

به جای RRF، میانگین وزن‌دار امتیازهای نرمال‌شده، یعنی Min-Max، را امتحان کنید. با وزن‌های ۰٫۳، ۰٫۵ و ۰٫۷ برای جستجوی معنایی، کدام بهتر است؟

</div>

In [ ]:
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


def weighted(alpha):
    return lambda q: rank(alpha * minmax(dense_scores(q)) + (1 - alpha) * minmax(bm25_scores(q)))


methods = {"dense": lambda q: rank(dense_scores(q)), "BM25": lambda q: rank(bm25_scores(q)),
           "RRF": lambda q: rrf([rank(dense_scores(q)), rank(bm25_scores(q))])}
methods.update({f"weighted α={a}": weighted(a) for a in (0.3, 0.5, 0.7)})
pd.DataFrame({name: evaluate_retrieval(fn) for name, fn in methods.items()}).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

ترکیب وزن‌دار یک پارامتر دارد، $\alpha$، که باید تنظیم شود. اگر آن را روی همین ۳۰ پرسش تنظیم کنیم، نتیجهٔ بهترین $\alpha$ خوش‌بینانه است. همان مشکل تنظیم روی دادهٔ آزمون که در جلسهٔ ۴ دیدیم. Min-Max هم به امتیازهای کرانهٔ هر پرسش حساس است: اگر یک تکه امتیاز BM25 خیلی بالاتری داشته باشد، بقیه به صفر نزدیک می‌شوند.

RRF پارامتر یادگرفتنی ندارد و به مقیاس امتیازها حساس نیست. به همین دلیل نقطهٔ شروع پیش‌فرض خوبی است. اگر دادهٔ ارزیابی کافی دارید، ترکیب وزن‌دار را روی بخش Dev تنظیم و روی Test گزارش کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

ده پرسش جدید و دشوارتر، مثلاً ترکیبی یا محاوره‌ای، به مجموعه ارزیابی اضافه کنید و تمام ارزیابی را تکرار کنید. رتبه روش‌ها تغییر می‌کند؟

</div>

In [ ]:
hard = [
    ("اگه سفارشم برگشت خورد به انبار، پولم چی میشه؟", "به کیف پول مشتری بازگردانده می‌شود"),
    ("ساختمون ما آسانسور نداره و طبقه پنجمیم، یخچال رو بالا میارید؟", "بالاتر از طبقه سوم هزینه حمل جداگانه"),
    ("کارت هدیه رو میشه نقد کرد؟", "قابل تبدیل به وجه نقد نیست"),
    ("شارژ کیف پولمو میتونم برگردونم به حسابم؟", "قابل برداشت به حساب بانکی است"),
    ("خودم کولرو نصب کنم گارانتیش میپره؟", "توسط فردی غیر از نصاب مجاز نصب شود"),
    ("شماره کارتمو جایی ذخیره می‌کنید؟", "اطلاعات کارت بانکی مشتریان در سرورهای نوین‌کالا ذخیره نمی‌شود"),
    ("یکی زنگ زده از طرف شما کد پیامکو خواسته، بدم؟", "هرگز رمز یک‌بار مصرف یا اطلاعات کارت بانکی را تلفنی"),
    ("امتیاز خریدم از کی حساب میشه؟", "پس از پایان مهلت هفت‌روزه بازگشت کالا فعال می‌شود"),
    ("اگه جنسی که با کد تخفیف خریدم رو پس بدم، کدم برمی‌گرده؟", "کد تخفیف دوباره فعال نمی‌شود"),
    ("سازمان دولتی هستیم، میشه چک بدیم؟", "مبلغ را به صورت چک پرداخت کنند"),
]
hard_set = [{"question": q, "evidence": e} for q, e in hard]
for item in hard_set:                      # every evidence sentence must point to exactly one chunk
    assert sum(normalize(item["evidence"]) in c["text"] for c in chunks) == 1, item

rows = {}
for name, fn in [("dense", methods["dense"]), ("BM25", methods["BM25"]), ("RRF", methods["RRF"])]:
    rows[(name, "original 30")] = evaluate_retrieval(fn)
    rows[(name, "hard 10")] = evaluate_retrieval(fn, hard_set)
pd.DataFrame(rows).T.round(3)

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

پرسش‌های جدید محاوره‌ای‌اند و کلمه‌هایشان با سند فرق دارد: «نقد کرد» به جای «تبدیل به وجه نقد»، «میپره» به جای «باطل می‌شود». انتظار داریم BM25 روی این مجموعه افت بیشتری از جستجوی معنایی داشته باشد. اگر رتبهٔ روش‌ها روی دو مجموعه فرق کرد، یعنی مجموعهٔ اولیه نمایندهٔ خوبی از پرسش‌های واقعی کاربران نبوده است.

بررسی `assert` در کد مهم است: پیش از اضافه کردن پرسش به مجموعهٔ ارزیابی، مطمئن شوید عبارت شاهد واقعاً و فقط در یک تکه وجود دارد. عبارت شاهد اشتباه، ارزیابی را بی‌صدا خراب می‌کند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

پنج پاسخ را خودتان از نظر وفاداری بررسی کنید و با نظر داور مقایسه کنید. داور کجا اشتباه کرده است؟

</div>

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Faithfulness(BaseModel):
    unsupported_claims: list[str]
    faithful: bool


RAG_SYSTEM = """You are the customer-support assistant of Novin Kala. Answer in Persian using ONLY the numbered
sources. End every factual sentence with its source number, e.g. [2]. If the sources do not contain the answer,
reply exactly: «متأسفانه در اسناد نوین‌کالا پاسخی برای این پرسش پیدا نکردم.»"""
FAITHFULNESS_RUBRIC = """List every factual claim in the ANSWER that is NOT stated in the SOURCES (ignore greetings
and the refusal sentence). Then set faithful=true only if the list is empty."""

rows = []
for item in eval_set[:5]:
    top = rrf([rank(dense_scores(item["question"])), rank(bm25_scores(item["question"]))])[:4]
    sources = "\n\n".join(f"<source id=\"{n}\">\n{chunks[i]['text']}\n</source>" for n, i in enumerate(top, 1))
    answer = chat(client, MODEL, f"{sources}\n\nCustomer question: {item['question']}", system=RAG_SYSTEM, temperature=0)
    response = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role": "system", "content": FAITHFULNESS_RUBRIC},
                  {"role": "user", "content": f"SOURCES:\n{sources}\n\nANSWER: {answer}"}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "Faithfulness", "schema": Faithfulness.model_json_schema()}})
    verdict = Faithfulness.model_validate_json(response.choices[0].message.content)
    rows.append({"question": item["question"], "answer": answer, "judge_faithful": verdict.faithful,
                 "unsupported": verdict.unsupported_claims, "sources": sources})
review = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 100)
review[["question", "answer", "judge_faithful", "unsupported"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

هر پاسخ را با منابعش مقایسه کنید (`print(review.sources[i])`) و نظر خودتان را در فهرست زیر بنویسید. مقدارهای فعلی فقط نمونه‌اند.

</div>

In [ ]:
my_faithful = [True, True, True, False, True]        # replace with your own judgement

review["mine"] = my_faithful
review["agree"] = review.mine == review.judge_faithful
print(f"agreement: {review.agree.mean():.0%}")
review.loc[~review.agree, ["question", "answer", "judge_faithful", "mine", "unsupported"]]

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

خطاهای رایج داور وفاداری:

- **سخت‌گیری بیش از حد:** بازنویسی یا خلاصه‌کردن درست منبع را «ادعای بی‌پشتوانه» می‌داند. مثلاً منبع می‌گوید «سه تا پنج روز کاری» و پاسخ می‌گوید «حداکثر پنج روز کاری».
- **سهل‌گیری:** ادعایی را که به دانش عمومی نزدیک است، مثل «جمعه تعطیل است»، بدون بررسی منبع می‌پذیرد.
- **نادیده گرفتن شرط‌ها:** پاسخی که شرط مهمی را حذف کرده، مثلاً «فقط برای کالاهای پلمب»، از نظر فنی ادعای غلطی ندارد و وفادار شمرده می‌شود، ولی گمراه‌کننده است. این خطا را معیار **کامل بودن** می‌سنجد، نه وفاداری.

هر جا با داور مخالف بودید، معیار را دقیق‌تر کنید یا نمونهٔ مرزی را به‌عنوان مثال به پرامپت داور اضافه کنید.

</div>